In [1]:
import utils
import os

np = utils.importer('numpy')
plt = utils.importer('matplotlib.pyplot')
pickle = utils.importer('pickle')
meshio = utils.importer('meshio')
Silhouette, Landscape = utils.importer('gudhi.representations', 'Silhouette', 'Landscape')
LogisticRegression = utils.importer('sklearn.linear_model', 'LogisticRegression')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.inspection import DecisionBoundaryDisplay

In [2]:
with open('E:/TDA/male.bin', 'rb') as fp:
    all_scans = pickle.load(fp)
    
male_nb_scans = len(all_scans)

In [3]:
with open('E:/TDA/female.bin', 'rb') as fp:
    all_scans += pickle.load(fp)

nb_scans = len(all_scans)

In [4]:
def plot_scan_3D_interactive(i):
    scan = all_scans[i]
    points_set = utils.point_set(scan)
    points_set.plot(color='green', opacity=0.5, size=2)

In [5]:
with open ('./Data/XY_trunk_decolor.bin', 'rb') as fp:
    XY_decolor = pickle.load(fp)

with open ('./Data/XX_trunk_decolor.bin', 'rb') as fp:
    XX_decolor = pickle.load(fp)

X_decolor = XY_decolor + XX_decolor
del XX_decolor, XY_decolor

In [6]:
test0 = [xd[0][:-1] for xd in X_decolor]
test1 = [xd[1] for xd in X_decolor]
test2 = [xd[2] for xd in X_decolor]

In [7]:
n0 = n1 = n2 = 1000
weight = lambda x: 1/(x[1]-x[0])
SH0 = Silhouette(resolution=n0, weight=weight)
sh0 = SH0.fit_transform(test0)

SH1 = Silhouette(resolution=n1, weight=weight)
sh1 = SH1.fit_transform(test1)

SH2 = Silhouette(resolution=n2, weight=weight)
sh2 = SH2.fit_transform(test2)

In [8]:
n = 10
m = (n0+n1+n2)//n

temp = np.array([list(x0) + list(x1) + list(x2) for x0, x1, x2 in zip(sh0, sh1, sh2)])
X = np.array([[np.mean(x[n*i:n*i+n]) for i in range(m)] for x in temp])
X = (X - X.mean()) / X.std()

y = [i < male_nb_scans for i in range(nb_scans)]


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/3, random_state=42)

In [10]:
names = [
    "Nearest Neighbors",
    "Linear SVM",
    "Gaussian Process",
    "Decision Tree",
    "Random Forest",
    "Neural Net",
    "AdaBoost",
    "Logistic Regression"
]

classifiers = [
    KNeighborsClassifier(),
    SVC(kernel="linear", random_state=42),
    GaussianProcessClassifier(random_state=42),
    DecisionTreeClassifier(random_state=42),
    RandomForestClassifier(n_estimators=150, random_state=42),
    MLPClassifier(hidden_layer_sizes=(150, 100, 50), random_state=42, max_iter=10000),
    AdaBoostClassifier(random_state=42),
    LogisticRegression(random_state=42, max_iter=10000)
]

In [11]:
scores = []

for name, clf in zip(names, classifiers):
    clf.fit(X_train, y_train)
    score = clf.score(X_test, y_test)
    score_train = clf.score(X_train, y_train)
    scores.append(score)
    print(name, ' -  Score:', score, ' -  STRAIN:', score_train)

Nearest Neighbors  -  Score: 0.9468503937007874  -  STRAIN: 0.9542322834645669
Linear SVM  -  Score: 0.9114173228346457  -  STRAIN: 0.9192913385826772
Gaussian Process  -  Score: 0.9468503937007874  -  STRAIN: 0.984251968503937
Decision Tree  -  Score: 0.8996062992125984  -  STRAIN: 1.0
Random Forest  -  Score: 0.9478346456692913  -  STRAIN: 1.0
Neural Net  -  Score: 0.9566929133858267  -  STRAIN: 0.9926181102362205
AdaBoost  -  Score: 0.9320866141732284  -  STRAIN: 0.9552165354330708
Logistic Regression  -  Score: 0.9015748031496063  -  STRAIN: 0.906988188976378


In [12]:
np.mean(scores)

0.9303641732283464